## Preprocess data

This notebook expects to take standard input data from the corridor model and analytics pipelines and output tables that are ready for the front end to use. Most notably this includes creating regional clusters and tagging each asset to them

### Import necessary functions

In [163]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [164]:
import os

### Little hack to reproducibly set working dir as the project directory (one level up) in notebooks
if "original_dir" not in vars():
    original_dir = os.path.abspath("")

os.chdir(original_dir)
os.chdir("..")  ### Adjust as needed to get to root
print(f"Project root (ensure this is correct): {os.getcwd()}")

import src.preprocessing.data_preprocessing as funcs


import geopandas as gpd
import pandas as pd
import numpy as np
import random
import sqlite3
from shapely import wkt
import re

from src.general_utilities import io_utils, logging_utils

random.seed(42)

pd.set_option("display.max_columns", 999)

Project root (ensure this is correct): /Users/Jake_Lieberfarb/projects/pea/vegx_thailand_front_end


#### Make DB Connection

In [165]:
conn = sqlite3.connect("data/F0_raw/digital_twin_database.sqlite")

#### Areas of Jurisdiction

In [166]:
aoj = io_utils.load_file_from_catalog("areas_of_jurisdiction")
aoj["NAME"] = aoj["NAME"].map(
    lambda x: re.sub(r"\.", "", x)
)  # periods in names can cause issues
aoj.head(3)

,NAME,CODE,CREATIONUSER,DATECREATED,LASTUSER,DATEMODIFIED,WORKREQUESTID,DESIGNID,WORKLOCATIONID,WORKFLOWSTATUS,WORKFUNCTION,AREA_CODE,SHAPE_Length,SHAPE_Area,geometry,layer
0,กฟจพังงา,1105101,495269,2022-02-10 00:00:00+00:00,495269,2021-01-18 00:00:00+00:00,None,,None,NaN,NaN,42,180308.267542,5.477385e+08,"MULTIPOLYGON (((448581.706 957501.162, 448717....",lb_aoj
1,กฟอตะกั่วป่า,1112101,495269,2022-02-10 00:00:00+00:00,495269,2021-06-10 00:00:00+00:00,None,,None,NaN,NaN,42,166760.746506,4.360139e+08,"MULTIPOLYGON (((419520.563 996670.750, 419836....",lb_aoj
2,กฟอห้วยยอด,1117101,495269,2022-02-10 00:00:00+00:00,495269,2022-09-19 00:00:00+00:00,None,,None,NaN,NaN,42,186099.474864,7.519688e+08,"MULTIPOLYGON (((579220.877 879361.122, 579310....",lb_aoj


In [167]:
aoj[["NAME"]].to_sql("areas_of_jurisdiction", conn, if_exists="replace", index=False)

79

## Create the output table for each solution

We'll want tables that the app needs stored in the db and tables that our pipeline will need stored as flat files via the catalog

### Veg Management

Veg comes with a few files:
* The corridor definitions
* The units of work aggregation
* The optimized trimming cycles to aim for

We need all of these in the database.

**Temporary solution**
We need to revisit the vegX pipeline itself to output a nice clean file format by default so this data wrangling here is not needed.

In [168]:
corridors_df = io_utils.load_file_from_catalog("front_end_input")
print(corridors_df.shape)
corridors_df.head(3)

(1646, 19)


,nearest_upstream_device,feeder_id_traced,line_type,corridor_length,number_lines,line_geometries,outages_lag_1,outages_lag_2,outages_lag_3,density_mean,nearest_upstream_device_list,health_score,adjusted_area_customers,density_distribution_meta,trimming_cost_meta,trimming_cost_mjm,raw_customers,criticality,risk
0,4284RC000000004,PPA04,overhead,3694.231633,77,"MULTILINESTRING ((533109.439 1003382.343, 5331...",0.0,2.0,3.0,0.036629,"['42SWKA000044732', '42SWKA000015942', '42SWKA...",0.129009,4011.779286,"{'high': '0.05 km', 'light': '2.09 km', 'mediu...",4410.226067,8239.334411,2480.0,4011.779286,342.468963
1,4284RC000000007,PPA02,overhead,11567.703408,107,"MULTILINESTRING ((529049.516 995943.404, 52904...",0.0,0.0,0.0,0.206048,"['4284SW000000840', '4284SW000000843', '4284SW...",0.171120,3406.186708,"{'high': '3.08 km', 'light': '0.71 km', 'mediu...",30254.349930,28061.268208,2135.0,3406.186708,996.932695
2,4284RC000000008,PPA09,overhead,8828.984783,77,"MULTILINESTRING ((522013.766 1008780.744, 5220...",0.0,0.0,0.0,0.042643,"['42SWKA000022484', '4284SW000000680', '4284SW...",0.187045,10014.266398,"{'high': '0.11 km', 'light': '4.56 km', 'mediu...",13409.299494,13902.366657,3359.0,10014.266398,2931.004799


In [169]:
corridors_df = corridors_df.rename(columns={"line_geometries": "geometry"})  # temp fix
corridors_df = gpd.GeoDataFrame(corridors_df, geometry="geometry")
corridors_df.head()

,nearest_upstream_device,feeder_id_traced,line_type,corridor_length,number_lines,geometry,outages_lag_1,outages_lag_2,outages_lag_3,density_mean,nearest_upstream_device_list,health_score,adjusted_area_customers,density_distribution_meta,trimming_cost_meta,trimming_cost_mjm,raw_customers,criticality,risk
0,4284RC000000004,PPA04,overhead,3694.231633,77,"MULTILINESTRING ((533109.439 1003382.343, 5331...",0.0,2.0,3.0,0.036629,"['42SWKA000044732', '42SWKA000015942', '42SWKA...",0.129009,4011.779286,"{'high': '0.05 km', 'light': '2.09 km', 'mediu...",4410.226067,8239.334411,2480.0,4011.779286,342.468963
1,4284RC000000007,PPA02,overhead,11567.703408,107,"MULTILINESTRING ((529049.516 995943.404, 52904...",0.0,0.0,0.0,0.206048,"['4284SW000000840', '4284SW000000843', '4284SW...",0.171120,3406.186708,"{'high': '3.08 km', 'light': '0.71 km', 'mediu...",30254.349930,28061.268208,2135.0,3406.186708,996.932695
2,4284RC000000008,PPA09,overhead,8828.984783,77,"MULTILINESTRING ((522013.766 1008780.744, 5220...",0.0,0.0,0.0,0.042643,"['42SWKA000022484', '4284SW000000680', '4284SW...",0.187045,10014.266398,"{'high': '0.11 km', 'light': '4.56 km', 'mediu...",13409.299494,13902.366657,3359.0,10014.266398,2931.004799
3,4284RC000000009,PPA09,overhead,12617.655355,91,"MULTILINESTRING ((527498.221 1015116.427, 5274...",0.0,1.0,2.0,0.054961,"['42RCKA000005755', '42SWKA000006256', '4284SW...",0.160003,8360.263959,"{'high': '1.35 km', 'light': '3.25 km', 'mediu...",25814.798308,18883.733605,1844.0,8360.263959,1631.271016
4,4284RC000000010,PPA01,overhead,9531.797185,66,"MULTILINESTRING ((512881.545 1011524.248, 5129...",3.0,4.0,6.0,0.163528,"['4284SW000000874', '4284SW000000829', '4284SW...",0.783438,7470.124799,"{'high': '3.22 km', 'light': '1.99 km', 'mediu...",24543.456998,4734.557821,2632.0,7470.124799,23412.464310


In [170]:
veg_df = funcs.create_features_for_veg(corridors_df)
veg_df["corridor length (km)"] = veg_df["corridor length"]
veg_df["corridor length (km)"] = veg_df["corridor length (km)"] / 1000
veg_df.head()

,nearest upstream device,feeder id,line type,corridor length,number lines,geometry,count of outages last year,count of outages last two years,count of outages last three years,average vegetation density (%) [META + Sentinel-2],nearest_upstream_device_list,probability of outage (%),customers affected (adjusted),density_distribution_meta,cost to trim (BHT) [META + Sentinel-2],cost to trim (BHT) [MJM],customers affected (original),criticality,risk (customer interruptions),corridor length (km)
0,4284RC000000004,PPA04,overhead,3694.23,77,"MULTILINESTRING ((533109.439 1003382.343, 5331...",0.0,2.0,3.0,0.04,"['42SWKA000044732', '42SWKA000015942', '42SWKA...",0.13,4011.78,"{'high': '0.05 km', 'light': '2.09 km', 'mediu...",4410.23,8239.33,2480.0,4011.78,342.47,3.69423
1,4284RC000000007,PPA02,overhead,11567.70,107,"MULTILINESTRING ((529049.516 995943.404, 52904...",0.0,0.0,0.0,0.21,"['4284SW000000840', '4284SW000000843', '4284SW...",0.17,3406.19,"{'high': '3.08 km', 'light': '0.71 km', 'mediu...",30254.35,28061.27,2135.0,3406.19,996.93,11.56770
2,4284RC000000008,PPA09,overhead,8828.98,77,"MULTILINESTRING ((522013.766 1008780.744, 5220...",0.0,0.0,0.0,0.04,"['42SWKA000022484', '4284SW000000680', '4284SW...",0.19,10014.27,"{'high': '0.11 km', 'light': '4.56 km', 'mediu...",13409.30,13902.37,3359.0,10014.27,2931.00,8.82898
3,4284RC000000009,PPA09,overhead,12617.66,91,"MULTILINESTRING ((527498.221 1015116.427, 5274...",0.0,1.0,2.0,0.05,"['42RCKA000005755', '42SWKA000006256', '4284SW...",0.16,8360.26,"{'high': '1.35 km', 'light': '3.25 km', 'mediu...",25814.80,18883.73,1844.0,8360.26,1631.27,12.61766
4,4284RC000000010,PPA01,overhead,9531.80,66,"MULTILINESTRING ((512881.545 1011524.248, 5129...",3.0,4.0,6.0,0.16,"['4284SW000000874', '4284SW000000829', '4284SW...",0.78,7470.12,"{'high': '3.22 km', 'light': '1.99 km', 'mediu...",24543.46,4734.56,2632.0,7470.12,23412.46,9.53180


### Create bins 

In [171]:
columns_to_categorize = [
    "probability of outage (%)",  # probability of failure
    "risk (customer interruptions)",  # risk
    "customers affected (adjusted)",  # criticality
]

for column in columns_to_categorize:
    veg_df = funcs.categorize_by_deciles(df=veg_df, column=column)
veg_df.head()

,nearest upstream device,feeder id,line type,corridor length,number lines,geometry,count of outages last year,count of outages last two years,count of outages last three years,average vegetation density (%) [META + Sentinel-2],nearest_upstream_device_list,probability of outage (%),customers affected (adjusted),density_distribution_meta,cost to trim (BHT) [META + Sentinel-2],cost to trim (BHT) [MJM],customers affected (original),criticality,risk (customer interruptions),corridor length (km),probability of outage (%) [bins],risk (customer interruptions) [bins],customers affected (adjusted) [bins]
0,4284RC000000004,PPA04,overhead,3694.23,77,"MULTILINESTRING ((533109.439 1003382.343, 5331...",0.0,2.0,3.0,0.04,"['42SWKA000044732', '42SWKA000015942', '42SWKA...",0.13,4011.78,"{'high': '0.05 km', 'light': '2.09 km', 'mediu...",4410.23,8239.33,2480.0,4011.78,342.47,3.69423,medium,high,high
1,4284RC000000007,PPA02,overhead,11567.70,107,"MULTILINESTRING ((529049.516 995943.404, 52904...",0.0,0.0,0.0,0.21,"['4284SW000000840', '4284SW000000843', '4284SW...",0.17,3406.19,"{'high': '3.08 km', 'light': '0.71 km', 'mediu...",30254.35,28061.27,2135.0,3406.19,996.93,11.56770,medium,high,high
2,4284RC000000008,PPA09,overhead,8828.98,77,"MULTILINESTRING ((522013.766 1008780.744, 5220...",0.0,0.0,0.0,0.04,"['42SWKA000022484', '4284SW000000680', '4284SW...",0.19,10014.27,"{'high': '0.11 km', 'light': '4.56 km', 'mediu...",13409.30,13902.37,3359.0,10014.27,2931.00,8.82898,medium,high,high
3,4284RC000000009,PPA09,overhead,12617.66,91,"MULTILINESTRING ((527498.221 1015116.427, 5274...",0.0,1.0,2.0,0.05,"['42RCKA000005755', '42SWKA000006256', '4284SW...",0.16,8360.26,"{'high': '1.35 km', 'light': '3.25 km', 'mediu...",25814.80,18883.73,1844.0,8360.26,1631.27,12.61766,medium,high,high
4,4284RC000000010,PPA01,overhead,9531.80,66,"MULTILINESTRING ((512881.545 1011524.248, 5129...",3.0,4.0,6.0,0.16,"['4284SW000000874', '4284SW000000829', '4284SW...",0.78,7470.12,"{'high': '3.22 km', 'light': '1.99 km', 'mediu...",24543.46,4734.56,2632.0,7470.12,23412.46,9.53180,high,high,high


#### Add in scenario outcomes

In [172]:
import re

scenario_outcomes = io_utils.load_file_from_catalog("optimization_scenarios")

scenario_outcomes = scenario_outcomes[
    ["nearest_upstream_device", "scenario", "frequency"]
].rename(columns={"nearest_upstream_device": "nearest upstream device"})
scenario_outcomes = (
    scenario_outcomes.groupby(["nearest upstream device", "scenario"])["frequency"]
    .agg("first")
    .unstack()
)
scenario_outcomes.columns = [
    re.sub("_", " ", x) + " frequency" for x in scenario_outcomes.columns
]
scenario_outcomes = scenario_outcomes.reset_index()
scenario_outcomes = scenario_outcomes.rename(
    columns={"hybrid frequency": "chosen frequency"}
)
scenario_outcomes.head(3)

,nearest upstream device,chosen frequency,scenario 1 frequency,scenario 2 frequency
0,4284RC000000004,T4,T4,T4
1,4284RC000000007,T3,T4,T1
2,4284RC000000008,T4,T4,T4


In [173]:
veg_df = veg_df.merge(scenario_outcomes, how="left", on="nearest upstream device")
veg_df.head(3)

,nearest upstream device,feeder id,line type,corridor length,number lines,geometry,count of outages last year,count of outages last two years,count of outages last three years,average vegetation density (%) [META + Sentinel-2],nearest_upstream_device_list,probability of outage (%),customers affected (adjusted),density_distribution_meta,cost to trim (BHT) [META + Sentinel-2],cost to trim (BHT) [MJM],customers affected (original),criticality,risk (customer interruptions),corridor length (km),probability of outage (%) [bins],risk (customer interruptions) [bins],customers affected (adjusted) [bins],chosen frequency,scenario 1 frequency,scenario 2 frequency
0,4284RC000000004,PPA04,overhead,3694.23,77,"MULTILINESTRING ((533109.439 1003382.343, 5331...",0.0,2.0,3.0,0.04,"['42SWKA000044732', '42SWKA000015942', '42SWKA...",0.13,4011.78,"{'high': '0.05 km', 'light': '2.09 km', 'mediu...",4410.23,8239.33,2480.0,4011.78,342.47,3.69423,medium,high,high,T4,T4,T4
1,4284RC000000007,PPA02,overhead,11567.70,107,"MULTILINESTRING ((529049.516 995943.404, 52904...",0.0,0.0,0.0,0.21,"['4284SW000000840', '4284SW000000843', '4284SW...",0.17,3406.19,"{'high': '3.08 km', 'light': '0.71 km', 'mediu...",30254.35,28061.27,2135.0,3406.19,996.93,11.56770,medium,high,high,T3,T4,T1
2,4284RC000000008,PPA09,overhead,8828.98,77,"MULTILINESTRING ((522013.766 1008780.744, 5220...",0.0,0.0,0.0,0.04,"['42SWKA000022484', '4284SW000000680', '4284SW...",0.19,10014.27,"{'high': '0.11 km', 'light': '4.56 km', 'mediu...",13409.30,13902.37,3359.0,10014.27,2931.00,8.82898,medium,high,high,T4,T4,T4


In [174]:
# map the names to frequncies

# Mapping dictionary
frequency_mapping = {
    "T1": "three times a year",
    "T2": "three times a year",
    "T3": "two time a year",
    "T4": "one time a year",
    "T5": "one time a year",
}

# Apply mapping to each frequency column
for col in ["chosen frequency", "scenario 1 frequency", "scenario 2 frequency"]:
    veg_df[col] = veg_df[col].map(frequency_mapping)

veg_df.head()

,nearest upstream device,feeder id,line type,corridor length,number lines,geometry,count of outages last year,count of outages last two years,count of outages last three years,average vegetation density (%) [META + Sentinel-2],nearest_upstream_device_list,probability of outage (%),customers affected (adjusted),density_distribution_meta,cost to trim (BHT) [META + Sentinel-2],cost to trim (BHT) [MJM],customers affected (original),criticality,risk (customer interruptions),corridor length (km),probability of outage (%) [bins],risk (customer interruptions) [bins],customers affected (adjusted) [bins],chosen frequency,scenario 1 frequency,scenario 2 frequency
0,4284RC000000004,PPA04,overhead,3694.23,77,"MULTILINESTRING ((533109.439 1003382.343, 5331...",0.0,2.0,3.0,0.04,"['42SWKA000044732', '42SWKA000015942', '42SWKA...",0.13,4011.78,"{'high': '0.05 km', 'light': '2.09 km', 'mediu...",4410.23,8239.33,2480.0,4011.78,342.47,3.69423,medium,high,high,one time a year,one time a year,one time a year
1,4284RC000000007,PPA02,overhead,11567.70,107,"MULTILINESTRING ((529049.516 995943.404, 52904...",0.0,0.0,0.0,0.21,"['4284SW000000840', '4284SW000000843', '4284SW...",0.17,3406.19,"{'high': '3.08 km', 'light': '0.71 km', 'mediu...",30254.35,28061.27,2135.0,3406.19,996.93,11.56770,medium,high,high,two time a year,one time a year,three times a year
2,4284RC000000008,PPA09,overhead,8828.98,77,"MULTILINESTRING ((522013.766 1008780.744, 5220...",0.0,0.0,0.0,0.04,"['42SWKA000022484', '4284SW000000680', '4284SW...",0.19,10014.27,"{'high': '0.11 km', 'light': '4.56 km', 'mediu...",13409.30,13902.37,3359.0,10014.27,2931.00,8.82898,medium,high,high,one time a year,one time a year,one time a year
3,4284RC000000009,PPA09,overhead,12617.66,91,"MULTILINESTRING ((527498.221 1015116.427, 5274...",0.0,1.0,2.0,0.05,"['42RCKA000005755', '42SWKA000006256', '4284SW...",0.16,8360.26,"{'high': '1.35 km', 'light': '3.25 km', 'mediu...",25814.80,18883.73,1844.0,8360.26,1631.27,12.61766,medium,high,high,one time a year,one time a year,two time a year
4,4284RC000000010,PPA01,overhead,9531.80,66,"MULTILINESTRING ((512881.545 1011524.248, 5129...",3.0,4.0,6.0,0.16,"['4284SW000000874', '4284SW000000829', '4284SW...",0.78,7470.12,"{'high': '3.22 km', 'light': '1.99 km', 'mediu...",24543.46,4734.56,2632.0,7470.12,23412.46,9.53180,high,high,high,one time a year,one time a year,one time a year


#### Export to file

In [175]:
veg_out = veg_df.drop(
    [
        "line type",
        "number lines",
        "count of outages last year",
        "count of outages last two years",
        "count of outages last three years",
        "criticality",
    ],
    axis=1,
)
veg_out.head()

,nearest upstream device,feeder id,corridor length,geometry,average vegetation density (%) [META + Sentinel-2],nearest_upstream_device_list,probability of outage (%),customers affected (adjusted),density_distribution_meta,cost to trim (BHT) [META + Sentinel-2],cost to trim (BHT) [MJM],customers affected (original),risk (customer interruptions),corridor length (km),probability of outage (%) [bins],risk (customer interruptions) [bins],customers affected (adjusted) [bins],chosen frequency,scenario 1 frequency,scenario 2 frequency
0,4284RC000000004,PPA04,3694.23,"MULTILINESTRING ((533109.439 1003382.343, 5331...",0.04,"['42SWKA000044732', '42SWKA000015942', '42SWKA...",0.13,4011.78,"{'high': '0.05 km', 'light': '2.09 km', 'mediu...",4410.23,8239.33,2480.0,342.47,3.69423,medium,high,high,one time a year,one time a year,one time a year
1,4284RC000000007,PPA02,11567.70,"MULTILINESTRING ((529049.516 995943.404, 52904...",0.21,"['4284SW000000840', '4284SW000000843', '4284SW...",0.17,3406.19,"{'high': '3.08 km', 'light': '0.71 km', 'mediu...",30254.35,28061.27,2135.0,996.93,11.56770,medium,high,high,two time a year,one time a year,three times a year
2,4284RC000000008,PPA09,8828.98,"MULTILINESTRING ((522013.766 1008780.744, 5220...",0.04,"['42SWKA000022484', '4284SW000000680', '4284SW...",0.19,10014.27,"{'high': '0.11 km', 'light': '4.56 km', 'mediu...",13409.30,13902.37,3359.0,2931.00,8.82898,medium,high,high,one time a year,one time a year,one time a year
3,4284RC000000009,PPA09,12617.66,"MULTILINESTRING ((527498.221 1015116.427, 5274...",0.05,"['42RCKA000005755', '42SWKA000006256', '4284SW...",0.16,8360.26,"{'high': '1.35 km', 'light': '3.25 km', 'mediu...",25814.80,18883.73,1844.0,1631.27,12.61766,medium,high,high,one time a year,one time a year,two time a year
4,4284RC000000010,PPA01,9531.80,"MULTILINESTRING ((512881.545 1011524.248, 5129...",0.16,"['4284SW000000874', '4284SW000000829', '4284SW...",0.78,7470.12,"{'high': '3.22 km', 'light': '1.99 km', 'mediu...",24543.46,4734.56,2632.0,23412.46,9.53180,high,high,high,one time a year,one time a year,one time a year


### organize corridor view into organized columns 

In [176]:
veg_out = veg_out[
    [
        "feeder id",
        "nearest upstream device",
        "nearest_upstream_device_list",
        "geometry",
        "corridor length (km)",
        "average vegetation density (%) [META + Sentinel-2]",
        "density_distribution_meta",
        # "average vegetation density (%) [MJM]",
        "probability of outage (%)",
        "probability of outage (%) [bins]",
        "cost to trim (BHT) [META + Sentinel-2]",
        "cost to trim (BHT) [MJM]",
        "risk (customer interruptions)",
        "risk (customer interruptions) [bins]",
        "customers affected (original)",
        "customers affected (adjusted)",
        "customers affected (adjusted) [bins]",
        "scenario 1 frequency",
        "scenario 2 frequency",
        "chosen frequency",
    ]
].rename(
    columns={
        "density_distribution_meta": "density distribution meta",
        "average vegetation density (%) [META + Sentinel-2]": "vegetation density (%) [META + Sentinel-2]",
        "nearest_upstream_device_list": "nearest upstream device list",
        # "average vegetation density (%) [MJM]": "vegetation density (%) [MJM]",
        "scenario 1 frequency": "cost focus scenario frequency",
        "scenario 2 frequency": "reliability focus scenario frequency",
        "hybrid scenario frequency": "chosen scenario frequency",
    }
)

In [177]:
io_utils.save_file_via_catalog(veg_out, "veg_features")

#### Export corridor file to sqllite database

Fill in Nans, cast to basic types and stringify the geometry

It's basically the same table as the above though. I just keep the parquet around for convenience on the map generation pipeline. The app will use the DB.

In [178]:
# Select non-categorical columns
non_categorical_cols = veg_out.select_dtypes(exclude=["category"]).columns

# Fill NaN for non-categorical columns only
veg_out[non_categorical_cols] = veg_out[non_categorical_cols].fillna(0)

# Convert to the desired CRS
veg_corridors_to_sql = veg_out.to_crs("epsg:4326")

veg_corridors_to_sql["feeder id"] = veg_corridors_to_sql["feeder id"].apply(
    lambda x: str(x)
)
veg_corridors_to_sql["geometry_wkt"] = veg_corridors_to_sql["geometry"].map(
    lambda x: x.wkt
)
veg_corridors_to_sql = veg_corridors_to_sql.drop("geometry", axis=1)


veg_corridors_to_sql.map(lambda x: round(x, 2) if type(x) == float else x).to_sql(
    "veg_features", conn, if_exists="replace", index=False
)
veg_corridors_to_sql.head()

,feeder id,nearest upstream device,nearest upstream device list,corridor length (km),vegetation density (%) [META + Sentinel-2],density distribution meta,probability of outage (%),probability of outage (%) [bins],cost to trim (BHT) [META + Sentinel-2],cost to trim (BHT) [MJM],risk (customer interruptions),risk (customer interruptions) [bins],customers affected (original),customers affected (adjusted),customers affected (adjusted) [bins],cost focus scenario frequency,reliability focus scenario frequency,chosen frequency,geometry_wkt
0,PPA04,4284RC000000004,"['42SWKA000044732', '42SWKA000015942', '42SWKA...",3.69423,0.04,"{'high': '0.05 km', 'light': '2.09 km', 'mediu...",0.13,medium,4410.23,8239.33,342.47,high,2480.0,4011.78,high,one time a year,one time a year,one time a year,MULTILINESTRING ((99.30129312481573 9.07703173...
1,PPA02,4284RC000000007,"['4284SW000000840', '4284SW000000843', '4284SW...",11.56770,0.21,"{'high': '3.08 km', 'light': '0.71 km', 'mediu...",0.17,medium,30254.35,28061.27,996.93,high,2135.0,3406.19,high,one time a year,three times a year,two time a year,MULTILINESTRING ((99.26429937195249 9.00977533...
2,PPA09,4284RC000000008,"['42SWKA000022484', '4284SW000000680', '4284SW...",8.82898,0.04,"{'high': '0.11 km', 'light': '4.56 km', 'mediu...",0.19,medium,13409.30,13902.37,2931.00,high,3359.0,10014.27,high,one time a year,one time a year,one time a year,MULTILINESTRING ((99.20035113328484 9.12592949...
3,PPA09,4284RC000000009,"['42RCKA000005755', '42SWKA000006256', '4284SW...",12.61766,0.05,"{'high': '1.35 km', 'light': '3.25 km', 'mediu...",0.16,medium,25814.80,18883.73,1631.27,high,1844.0,8360.26,high,one time a year,two time a year,one time a year,MULTILINESTRING ((99.2503058998916 9.183204348...
4,PPA01,4284RC000000010,"['4284SW000000874', '4284SW000000829', '4284SW...",9.53180,0.16,"{'high': '3.22 km', 'light': '1.99 km', 'mediu...",0.78,high,24543.46,4734.56,23412.46,high,2632.0,7470.12,high,one time a year,one time a year,one time a year,MULTILINESTRING ((99.11724547060244 9.15078080...


#### Make a feeder level file

Here we do a groupby agg to get feeder level features for the first maps

#### Create length weighted score so we can give a sensible estimate of "mean" density for a feeder

In [179]:
veg_df["share line miles"] = veg_df["corridor length"] / veg_df.groupby("feeder id")[
    "corridor length"
].transform("sum")

veg_df["weighted density (%) [META + Sentinel-2]"] = (
    veg_df["average vegetation density (%) [META + Sentinel-2]"]
    * veg_df["share line miles"]
)

# veg_df["weighted density (%) [MJM]"] = (
#     veg_df["average vegetation density (%) [MJM]"] * veg_df["share line miles"]
# )

veg_df["weighted probability"] = (
    veg_df["probability of outage (%)"] * veg_df["share line miles"]
)

# for columns that have
veg_df["nearest_upstream_device_list"] = veg_df["nearest_upstream_device_list"].map(
    lambda x: re.sub(",", "", str(x))
)
veg_df["density_distribution_meta"] = veg_df["density_distribution_meta"].map(
    lambda x: re.sub(",", "", str(x))
)
veg_df.head()

,nearest upstream device,feeder id,line type,corridor length,number lines,geometry,count of outages last year,count of outages last two years,count of outages last three years,average vegetation density (%) [META + Sentinel-2],nearest_upstream_device_list,probability of outage (%),customers affected (adjusted),density_distribution_meta,cost to trim (BHT) [META + Sentinel-2],cost to trim (BHT) [MJM],customers affected (original),criticality,risk (customer interruptions),corridor length (km),probability of outage (%) [bins],risk (customer interruptions) [bins],customers affected (adjusted) [bins],chosen frequency,scenario 1 frequency,scenario 2 frequency,share line miles,weighted density (%) [META + Sentinel-2],weighted probability
0,4284RC000000004,PPA04,overhead,3694.23,77,"MULTILINESTRING ((533109.439 1003382.343, 5331...",0.0,2.0,3.0,0.04,['42SWKA000044732' '42SWKA000015942' '42SWKA00...,0.13,4011.78,{'high': '0.05 km' 'light': '2.09 km' 'medium'...,4410.23,8239.33,2480.0,4011.78,342.47,3.69423,medium,high,high,one time a year,one time a year,one time a year,0.062399,0.002496,0.008112
1,4284RC000000007,PPA02,overhead,11567.70,107,"MULTILINESTRING ((529049.516 995943.404, 52904...",0.0,0.0,0.0,0.21,['4284SW000000840' '4284SW000000843' '4284SW00...,0.17,3406.19,{'high': '3.08 km' 'light': '0.71 km' 'medium'...,30254.35,28061.27,2135.0,3406.19,996.93,11.56770,medium,high,high,two time a year,one time a year,three times a year,0.107317,0.022537,0.018244
2,4284RC000000008,PPA09,overhead,8828.98,77,"MULTILINESTRING ((522013.766 1008780.744, 5220...",0.0,0.0,0.0,0.04,['42SWKA000022484' '4284SW000000680' '4284SW00...,0.19,10014.27,{'high': '0.11 km' 'light': '4.56 km' 'medium'...,13409.30,13902.37,3359.0,10014.27,2931.00,8.82898,medium,high,high,one time a year,one time a year,one time a year,0.098836,0.003953,0.018779
3,4284RC000000009,PPA09,overhead,12617.66,91,"MULTILINESTRING ((527498.221 1015116.427, 5274...",0.0,1.0,2.0,0.05,['42RCKA000005755' '42SWKA000006256' '4284SW00...,0.16,8360.26,{'high': '1.35 km' 'light': '3.25 km' 'medium'...,25814.80,18883.73,1844.0,8360.26,1631.27,12.61766,medium,high,high,one time a year,one time a year,two time a year,0.141248,0.007062,0.022600
4,4284RC000000010,PPA01,overhead,9531.80,66,"MULTILINESTRING ((512881.545 1011524.248, 5129...",3.0,4.0,6.0,0.16,['4284SW000000874' '4284SW000000829' '4284SW00...,0.78,7470.12,{'high': '3.22 km' 'light': '1.99 km' 'medium'...,24543.46,4734.56,2632.0,7470.12,23412.46,9.53180,high,high,high,one time a year,one time a year,one time a year,0.042106,0.006737,0.032843


#### Create feeder level features

In [180]:
veg_units = veg_df[["feeder id", "geometry"]].dissolve("feeder id")

veg_units = veg_units.merge(
    veg_df.groupby("feeder id").agg(
        risk=pd.NamedAgg("risk (customer interruptions)", "sum"),
        average_probability_of_outage=pd.NamedAgg("weighted probability", "sum"),
        total_original_customer_count=pd.NamedAgg(
            "customers affected (original)", "max"
        ),
        total_adjusted_customer_count=pd.NamedAgg(
            "customers affected (adjusted)", "max"
        ),
        cost_to_trim_meta=pd.NamedAgg("cost to trim (BHT) [META + Sentinel-2]", "sum"),
        cost_to_trim_mjm=pd.NamedAgg("cost to trim (BHT) [MJM]", "sum"),
    ),
    left_index=True,
    right_index=True,
)

veg_units = veg_units.rename(
    columns={
        "risk": "risk (customer interruptions)",
        "average_probability_of_outage": "average probability of outage (%)",
        # "consequence_of_outage": "consequence of outage",
        "cost_to_trim_meta": "cost to trim (BHT) [META + Sentinel-2]",
        "cost_to_trim_mjm": "cost to trim (BHT) [MJM]",
        # "percent_vegetated": "Percent Vegetated [META + Sentinel-2]",
        "total_original_customer_count": "customers affected (original)",
        "total_adjusted_customer_count": "customers affected (adjusted)",
    }
)

veg_units["geometry"] = veg_units["geometry"].convex_hull
veg_units = veg_units.rename(columns={"geometry": "polygon"}).reset_index()
veg_units = gpd.GeoDataFrame(veg_units, geometry="polygon", crs=corridors_df.crs)
veg_units.head(3)

,feeder id,polygon,risk (customer interruptions),average probability of outage (%),customers affected (original),customers affected (adjusted),cost to trim (BHT) [META + Sentinel-2],cost to trim (BHT) [MJM]
0,PPA,"POLYGON ((528212.429 1004369.634, 528194.251 1...",0.00,0.020000,0.0,0.00,49.32,757.57
1,PPA01,"POLYGON ((520568.814 1000635.528, 520530.229 1...",43598.66,0.373581,7055.0,17342.37,621636.37,583361.66
2,PPA02,"POLYGON ((528501.448 991631.483, 526520.190 99...",9877.06,0.364357,3303.0,5662.45,309344.09,250241.83


#### Add in the jurisdiction

In [181]:
veg_units = veg_units.sjoin(
    aoj[["NAME", "geometry"]].rename(columns={"NAME": "Area of Jurisdiction"})
).drop("index_right", axis=1)
veg_units.head(3)

,feeder id,polygon,risk (customer interruptions),average probability of outage (%),customers affected (original),customers affected (adjusted),cost to trim (BHT) [META + Sentinel-2],cost to trim (BHT) [MJM],Area of Jurisdiction
0,PPA,"POLYGON ((528212.429 1004369.634, 528194.251 1...",0.00,0.020000,0.0,0.00,49.32,757.57,กฟสพุนพิน
1,PPA01,"POLYGON ((520568.814 1000635.528, 520530.229 1...",43598.66,0.373581,7055.0,17342.37,621636.37,583361.66,กฟสพุนพิน
1,PPA01,"POLYGON ((520568.814 1000635.528, 520530.229 1...",43598.66,0.373581,7055.0,17342.37,621636.37,583361.66,กฟสคีรีรัฐนิคม


#### Organize veg units columns 

In [182]:
veg_units = veg_units[
    [
        "Area of Jurisdiction",
        "feeder id",
        "polygon",
        "average probability of outage (%)",
        "cost to trim (BHT) [META + Sentinel-2]",
        "cost to trim (BHT) [MJM]",
        "customers affected (original)",
        "customers affected (adjusted)",
        "risk (customer interruptions)",
    ]
]

#### Save to file and DB

In [183]:
io_utils.save_file_via_catalog(
    veg_units,
    "veg_clusters",
)

veg_units_sql = veg_units.copy()
veg_units_sql["polygon"] = veg_units_sql["polygon"].map(lambda x: x.wkt)
veg_units_sql = veg_units_sql.fillna(0)
veg_units_sql.map(lambda x: round(x, 2) if type(x) == float else x).to_sql(
    "veg_units", conn, if_exists="replace", index=False
)

/var/folders/6v/2mx3m2gn0ks3l0g4vwxyq8g40000gp/T/ipykernel_66579/653799211.py:7: UserWarning: Geometry column does not contain geometry.
  veg_units_sql["polygon"] = veg_units_sql["polygon"].map(lambda x: x.wkt)


79

#### Scenario Outcomes

No need to do any transformations. Just put it in the db.

In [184]:
scenario_features = io_utils.load_file_from_catalog("optimization_metrics")
# Replace values in 'scenarios' column
scenario_features["scenario"] = scenario_features["scenario"].replace(
    {
        "hybrid": "chosen focus scenario",
        "scenario_1": "reliability focus scenario",
        "scenario_2": "cost focus scenario",
    }
)
print(scenario_features.shape)
scenario_features.head()

(3, 19)


,Next year Opex,Risk in system,Risk reduced by baseline,Achieved Risk by baseline,Risk reduced by optimization,Achieved Risk by optimization,Opex saved,Degradation time,Risk removed,Risk Reduction,Client Opex,Optimized Opex,Calculated Opex,% Risk reduced by baseline,% Risk reduced by optimization,Initial Saifi,Baseline Saifi,Optimized Saifi,scenario
0,None,430012.538552,86002.50771,344010.030842,100815.465383,329197.073169,2.358183e+06,0.5,0.4,0.5,1.572120e+07,1.336302e+07,1.572120e+07,20.0,23.444773,1.493524,1.194819,1.143371,chosen focus scenario
1,None,430012.538552,86002.50771,344010.030842,105955.281605,324057.256947,2.148347e+00,0.5,0.4,0.5,1.572120e+07,1.572120e+07,1.572120e+07,20.0,24.640045,1.493524,1.194819,1.125519,reliability focus scenario
2,None,430012.538552,86002.50771,344010.030842,87822.071360,342190.467192,5.502424e+06,0.5,0.4,0.5,1.572120e+07,1.021878e+07,1.572120e+07,20.0,20.423142,1.493524,1.194819,1.188499,cost focus scenario


In [185]:
scenario_features.to_sql("veg_scenarios", conn, if_exists="replace", index=False)

3

#### Shap

No need to do any transformations. Just put it in the db.

In [186]:
shap_features = io_utils.load_file_from_catalog("model_shap_features").rename(
    columns={"nearest_upstream_device": "nearest upstream device"}
)
print(shap_features.shape)
shap_features.head()

(1646, 72)


,atmospheric_pressure_max,atmospheric_pressure_mean,atmospheric_pressure_min,average_temp_mean,corridor_length,cum_year-2_number_point_trimming,cum_year-2_trim,cum_year-2_trimmed_in_season_dry_sum,cum_year-2_trimmed_in_season_humid_sum,cum_year-2_trimmed_km_sum,cum_year-3_number_point_trimming,cum_year-3_trim,cum_year-3_trimmed_in_season_dry_sum,cum_year-3_trimmed_in_season_humid_sum,cum_year-3_trimmed_km_sum,max_temp_max,min_temp_min,outages_lag_1,outages_lag_2,outages_lag_3,outages_lag_4,precipitation_max,precipitation_max_dry,precipitation_max_humid,precipitation_sum,precipitation_sum_dry,precipitation_sum_humid,tree_fall_outages_lag_1,tree_fall_outages_lag_2,tree_fall_outages_lag_3,tree_fall_outages_lag_4,tree_grow_in_outages_lag_1,tree_grow_in_outages_lag_2,tree_grow_in_outages_lag_3,tree_grow_in_outages_lag_4,wind_speed_max,wind_speed_mean,wind_speed_min,year-1_number_point_trimming,year-1_trim,year-1_trimmed_in_season_dry_sum,year-1_trimmed_in_season_humid_sum,year-1_trimmed_km_sum,years_since_last_trim,atmospheric_pressure_max_dry,atmospheric_pressure_max_humid,atmospheric_pressure_mean_dry,atmospheric_pressure_mean_humid,atmospheric_pressure_min_dry,atmospheric_pressure_min_humid,wind_speed_max_dry,wind_speed_max_humid,wind_speed_mean_dry,wind_speed_mean_humid,wind_speed_min_dry,wind_speed_min_humid,max_temp_max_dry,max_temp_max_humid,min_temp_min_dry,min_temp_min_humid,density_max,km_based_density_avg,height_max,km_based_height_avg,share_length_A,share_length_AA,share_length_ACSR,share_length_PIC,share_length_TAC,km_based_change_density_avg,change_density_max,nearest upstream device
0,0.0,0.024847,-0.001660,-0.042788,0.613367,-0.071769,-0.001253,-0.024912,0.018752,-0.002743,-0.010215,-0.007843,-0.004757,-0.013792,0.012264,-0.025911,-0.001952,0.381594,0.216512,0.0,0.0,-0.003097,-0.007685,-0.001631,-0.001247,0.000790,-0.000237,-0.035365,-0.000405,0.0,0.0,0.880518,0.032383,0.0,0.0,-0.000619,0.001651,0.031386,0.007657,-0.011463,-0.001035,-0.028777,0.007948,-0.001299,0.011953,0.003527,0.000138,-0.024844,0.001032,-0.021220,-0.003417,-0.007320,0.000229,0.002682,0.0,0.007999,0.002320,-0.062200,0.0,-0.007680,0.028097,-0.360120,0.066741,-0.157558,-0.000950,0.0,0.140931,-0.058058,0.0,-0.128256,0.027389,4284RC000000004
1,0.0,0.048889,-0.001018,-0.072569,0.049635,0.078027,0.021711,-0.032215,0.018901,-0.003465,0.050127,0.131355,-0.002277,0.009801,-0.013796,-0.046879,-0.007646,-0.084801,-0.026568,0.0,0.0,0.003366,-0.010392,-0.005599,-0.000608,-0.002739,-0.000238,-0.025083,-0.001369,0.0,0.0,-0.166092,-0.003578,0.0,0.0,0.000193,0.001290,0.028481,-0.028063,0.016713,0.001219,0.003818,-0.008213,0.016388,0.027820,0.004798,0.000138,-0.016150,-0.000233,-0.011742,-0.005100,-0.001598,0.002895,0.004112,0.0,0.009242,0.000375,-0.078391,0.0,-0.028356,-0.004229,0.070734,0.019762,-0.080484,0.002112,0.0,0.110058,0.216021,0.0,-0.167070,-0.026022,4284RC000000007
2,0.0,0.038909,-0.000703,-0.044829,0.390338,0.159223,0.021330,-0.011353,-0.023948,-0.003195,0.040110,0.139547,0.003086,0.019391,-0.016110,-0.033933,-0.001952,-0.085411,-0.020676,0.0,0.0,-0.021550,-0.011481,-0.002775,-0.002198,-0.002411,-0.000128,-0.024569,-0.001316,0.0,0.0,-0.170157,-0.003575,0.0,0.0,0.000193,0.001979,0.022582,0.029861,0.015333,0.006770,-0.019560,-0.012271,0.015156,0.014776,0.002432,0.000138,-0.024622,0.000250,-0.009435,-0.011541,-0.002470,0.000571,0.004529,0.0,0.007999,-0.007857,-0.097836,0.0,-0.010684,0.001002,-0.448000,0.046972,-0.061943,0.002536,0.0,0.064894,0.203000,0.0,-0.084504,0.016879,4284RC000000008
3,0.0,0.023169,-0.000703,-0.042146,0.049360,0.129468,0.021711,0.003954,-0.011107,-0.002996,-0.026382,0.084429,0.001328,0.023986,-0.002249,-0.030835,-0.001952,0.040060,0.183594,0.0,0.0,0.004134,-0.010005,-0.004056,-0.001456,0.000663,-0.000114,-0.018846,-0.000359,0.0,0.0,0.738284,0.030226,0.0,0.0,0.000193,0.002034,0.025347,0.021606,0.015333,0.012578,-0.018913,-0.004895,0.009753,0.014492,0.003606,0.000138,-0.014132,-0.000061,-0.009668,-0.005256,-0.007367,0.000571,0.004269,0

In [187]:
shap_features.to_sql("veg_shap_features", conn, if_exists="replace", index=False)

1646

#### Raw Asset Data

This is needed only for the purpose of making the shap charts

In [188]:
values_df = io_utils.load_file_from_catalog("model_inputs_prediction").rename(
    columns={"nearest_upstream_device": "nearest upstream device"}
)
values_df.head(1)

,nearest upstream device,feeder_id_traced,line_type,line_geometries,number_sac,number_acsr,number_pic,number_a,number_aa,number_xlpe,number_tac,number_oil,number_sm,number_lines,number_overhead,number_underground,number_eservice,number_lateral,number_main,number_insulated,number_not_insulated,number_steps_to_feeder,corridor_length,NUMBEROFUSER,C_transformer_count,P_transformer_count,C_transformer_load,P_transformer_load,downstream_NUMBEROFUSER,downstream_C_transformer_count,downstream_P_transformer_count,downstream_C_transformer_load,downstream_P_transformer_load,area_1_customer_count,area_2_customer_count,area_3_customer_count,area_4_customer_count,area_5_customer_count,length_eservice_overhead,length_overhead,length_L,length_M,length_insulated,length_not_insulated,length_A,length_AA,length_ACSR,length_PIC,length_SAC,length_TAC,total_line_length,share_length_eservice_overhead,share_length_overhead,share_length_L,share_length_M,share_length_insulated,share_length_not_insulated,share_length_A,share_length_AA,share_length_ACSR,share_length_PIC,share_length_SAC,share_length_TAC,number_poles,average_pole_height,min_pole_height,max_pole_height,pole_age,year,num_outages,outages_lag_1,outages_lag_2,outages_lag_3,outages_lag_4,target,num_tree_grow_in_outages,tree_grow_in_outages_lag_1,tree_grow_in_outages_lag_2,tree_grow_in_outages_lag_3,tree_grow_in_outages_lag_4,target_tree_grow_in,num_tree_fall_outages,tree_fall_outages_lag_1,tree_fall_outages_lag_2,tree_fall_outages_lag_3,tree_fall_outages_lag_4,target_fall_outages,num_outages_dry,outages_dry_lag_1,outages_dry_lag_2,outages_dry_lag_3,outages_dry_lag_4,num_tree_grow_in_outages_dry,tree_grow_in_outages_dry_lag_1,tree_grow_in_outages_dry_lag_2,tree_grow_in_outages_dry_lag_3,tree_grow_in_outages_dry_lag_4,num_tree_fall_outages_dry,tree_fall_outages_dry_lag_1,tree_fall_outages_dry_lag_2,tree_fall_outages_dry_lag_3,tree_fall_outages_dry_lag_4,num_outages_humid,outages_humid_lag_1,outages_humid_lag_2,outages_humid_lag_3,outages_humid_lag_4,num_tree_grow_in_outages_humid,tree_grow_in_outages_humid_lag_1,tree_grow_in_outages_humid_lag_2,tree_grow_in_outages_humid_lag_3,tree_grow_in_outages_humid_lag_4,num_tree_fall_outages_humid,tree_fall_outages_humid_lag_1,tree_fall_outages_humid_lag_2,tree_fall_outages_humid_lag_3,tree_fall_outages_humid_lag_4,density_max,density_min,density_mean,density_sum,height_max,change_density_max,change_density_mean,change_density_sum,km_based_change_density_avg,km_based_density_avg,km_based_height_avg,average_temp_mean,min_temp_min,max_temp_max,precipitation_sum,precipitation_max,wind_speed_min,wind_speed_max,wind_speed_mean,atmospheric_pressure_min,atmospheric_pressure_max,atmospheric_pressure_mean,atmospheric_pressure_max_dry,atmospheric_pressure_max_humid,atmospheric_pressure_mean_dry,atmospheric_pressure_mean_humid,atmospheric_pressure_min_dry,atmospheric_pressure_min_humid,average_temp_mean_dry,average_temp_mean_humid,max_temp_max_dry,max_temp_max_humid,min_temp_min_dry,min_temp_min_humid,precipitation_max_dry,precipitation_max_humid,precipitation_sum_dry,precipitation_sum_humid,wind_speed_max_dry,wind_speed_max_humid,wind_speed_mean_dry,wind_speed_mean_humid,wind_speed_min_dry,wind_speed_min_humid,trim,years_since_last_trim,last_trim_year,year-1_number_feeder_trimming,year-1_number_point_trimming,year-1_trimmed_km_sum,year-1_trimmed_in_season_dry_sum,year-1_trimmed_in_season_humid_sum,year-1_trim,cum_year-2_number_feeder_trimming,cum_year-3_number_feeder_trimming,cum_year-2_number_point_trimming,cum_year-3_number_point_trimming,cum_year-2_trimmed_km_sum,cum_year-3_trimmed_km_sum,cum_year-2_trimmed_in_season_dry_sum,cum_year-3_trimmed_in_season_dry_sum,cum_year-2_trimmed_in_season_humid_sum,cum_year-3_trimmed_in_season_humid_sum,cum_year-2_trim,cum_year-3_trim,static_mjm_density,actual_mjm_density
0,4284RC000000004,PPA04,overhead,"MULTILINESTRING ((533109.439 1003382.343, 5331...",76,1,0,0,0,0,0,0,0,91,77,0,14,32,45,77,0,2.0,3759.3742,371.0,5.0,8.0,857.5,2

In [189]:
values_df[list(shap_features)].to_sql("veg_raw", conn, if_exists="replace", index=False)

1646

#### Areas of Jursidction

In [190]:
# UAT test notebook ran
print("Test 1 passed")

Test 1 passed
